# Stimulus-Evoked BCIs — ERPs, the P300 Speller, and Single-Trial Classification

*Notebook #3 in the hands-on MNE series. Assumes the material of notebooks #1 (ERP averaging, filtering, epochs) and #2 (classification with cross-validation).*

Notebook #2 decoded *what the subject chose to think about* — left vs right motor imagery — by reading oscillatory power changes. This notebook explores the other major non-invasive BCI family: systems that decode *which external stimulus the subject is attending to*, using transient **event-related potentials (ERPs)**.

The centrepiece is the **P300 speller**, the first BCI that enabled locked-in patients to communicate letter by letter. The classification techniques developed here — temporal feature vectors, xDAWN spatial filtering, temporal decoding — apply identically to any ERP-based BCI.

## Table of contents

1. **Stimulus-evoked vs induced activity** — The fundamental distinction between this paradigm and motor imagery.
2. **The P300 component and the oddball paradigm** — Origin of the signal exploited by stimulus-evoked BCIs.
3. **The P300 speller** — Row/column flashing, target detection, and the Farwell–Donchin matrix.
4. **Steady-state visual evoked potentials (SSVEP)** — A brief note on the other major stimulus-evoked approach.
5. **Data preparation** — The MNE sample dataset repurposed for single-trial ERP classification.
6. **Single-trial variability** — Why individual epochs look nothing like the average.
7. **ERP classification: the vectorised approach** — Temporal waveforms as features, logistic regression as classifier.
8. **xDAWN spatial filtering** — The ERP analogue of CSP, maximising the signal-to-noise ratio of evoked responses.
9. **Temporal decoding** — At which latencies does the brain encode the stimulus category?
10. **From this analysis to a P300 speller** — How the same pipeline scales to a real communication system.

## Position in the textbook

- **Rao Ch. 9.1.4 — Stimulus-Evoked Potentials.** The physiological and paradigmatic content.
- **Rao Ch. 5 — Machine Learning.** Single-trial classification with regularised linear models.
- **Rao Ch. 12.1.5 — Restoring Communication.** The clinical application of P300 spellers.

## 1. Setup

The same stack as the previous notebooks. The MNE sample dataset is reused (no additional download required if notebook #1 has been run previously).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import mne
from mne.datasets import sample
from mne.decoding import Vectorizer, SlidingEstimator, GeneralizingEstimator, cross_val_multiscore

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

print("MNE:", mne.__version__)

## 2. Stimulus-evoked vs induced activity

EEG-based BCIs exploit two fundamentally different classes of brain signal. Understanding the distinction clarifies why the analysis pipelines differ between notebooks #2 and #3.

### 2.1 Induced activity (notebook #2)

Motor imagery produces **induced** oscillatory changes: the μ and β rhythms desynchronise. These changes are:

- **not phase-locked** to the cue — the timing of individual oscillation cycles varies from trial to trial;
- detectable only through **power** (amplitude envelope), not through raw waveform averaging;
- analysed via **band-power features** and spatial filters such as CSP.

Averaging raw waveforms across MI trials would produce approximately zero, because the oscillation phase is random relative to the cue.

### 2.2 Evoked activity (this notebook)

A sensory stimulus triggers a **transient, phase-locked** response — the ERP. Key properties:

- the waveform has a **consistent shape and latency** relative to stimulus onset;
- averaging across trials preserves the signal (because phases align), while noise cancels;
- the raw **temporal waveform** is itself the feature — no need to extract band-power.

The analysis pipeline is therefore different: instead of CSP on band-power, the approach uses the full time-domain waveform (or a spatially enhanced version of it) as input to a classifier.

| Property | Induced (MI) | Evoked (ERP) |
|---|---|---|
| Phase-locked to event? | No | Yes |
| Survives trial averaging? | Only the power envelope | Yes (the raw waveform) |
| Primary feature | Band-power (variance) | Temporal waveform |
| Canonical spatial filter | CSP | xDAWN |
| Example BCI paradigm | Motor imagery | P300 speller, SSVEP |

## 3. The P300 component and the oddball paradigm

The **P300** (or P3) is a positive ERP deflection occurring approximately 300 ms after a stimulus, maximal over centro-parietal electrodes. It was first described by Sutton et al. (1965) and has since become one of the most studied components in cognitive neuroscience.

### 3.1 The oddball paradigm

The P300 is most reliably elicited by the **oddball paradigm**:

1. Present a stream of stimuli at a regular pace (e.g., one per second).
2. Most stimuli belong to a **standard** (frequent) category.
3. Occasionally, a **deviant** (rare, task-relevant) stimulus appears.
4. The subject is instructed to attend to (or count, or press a button for) the deviant.

The deviant elicits a large P300; the standard does not. The amplitude of the P300 scales with the rarity and task-relevance of the deviant. Three sub-components are often distinguished:

- **P3a** — frontal, driven by stimulus novelty, occurs even without active attention;
- **P3b** — parietal, driven by task relevance, requires the subject's active engagement.

For BCI purposes, the **P3b** is the target component: its presence on a given trial signals that the flashed stimulus was the one the subject was attending to.

### 3.2 Why the P300 matters for BCIs

The P300 provides a binary decision per trial: *"Was the subject attending to this stimulus — yes or no?"* By flashing different options in sequence and asking the classifier to identify which flash elicited a P300, the system can infer the subject's intended choice without any motor output.

This insight led to the first practical communication BCI: the **P300 speller**.

## 4. The P300 speller

The P300 speller was introduced by Farwell and Donchin (1988) and remains one of the most successful non-invasive BCI paradigms for communication.

### 4.1 The spelling matrix

A **6 × 6 grid** of characters is displayed on a screen:

```
A  B  C  D  E  F
G  H  I  J  K  L
M  N  O  P  Q  R
S  T  U  V  W  X
Y  Z  1  2  3  4
5  6  7  8  9  0
```

The subject fixates on the letter to be communicated. Rows and columns of the matrix flash in random order. When the row or column containing the target letter flashes, the subject perceives a rare, task-relevant event — an oddball — and a P300 is elicited.

Since each letter lies at the intersection of exactly one row and one column, identifying the target row and the target column uniquely determines the intended letter. 12 flashes (6 rows + 6 columns) constitute one **flash sequence**; multiple sequences are typically averaged to improve accuracy.

### 4.2 The classification problem

For each flash, the classifier receives a short EEG epoch and must decide: **target or non-target?**

- In a single sequence of 12 flashes, exactly **2 are targets** (the row and column containing the desired letter) and **10 are non-targets**. This imbalance is a defining feature of P300 speller classification.
- After several sequences (typically 5–15), the system accumulates evidence and selects the row and column with the highest cumulative target probability.

The classification techniques developed in this notebook — temporal features, spatial filtering, regularised linear models — are exactly those used in competitive P300 speller implementations. The feature space is the ERP waveform; the classifier separates the small P300-bearing traces from the larger pool of non-target traces.

### 4.3 Performance

Typical spelling rates range from **1–8 characters per minute**, depending on the subject, the number of averaged sequences, and the classifier. State-of-the-art systems with adaptive classifiers and language models approach the upper end of this range. For users with locked-in syndrome, even 1 character per minute can be transformative.

## 5. A note on SSVEP

The other major stimulus-evoked BCI paradigm is the **Steady-State Visual Evoked Potential (SSVEP)**. Rather than detecting a transient P300, the approach exploits the brain's tendency to oscillate at the frequency of a flickering visual stimulus.

Each selectable option on screen flickers at a distinct frequency (e.g., 8 Hz, 10 Hz, 12 Hz, 15 Hz). The subject fixates on the desired option; the occipital EEG then shows a peak at the corresponding frequency and its harmonics. Frequency detection via FFT identifies the attended target.

SSVEP-based BCIs tend to achieve **higher information transfer rates** than P300 spellers (10–40 characters per minute in optimised systems), because frequency detection is more robust than single-trial ERP classification. The trade-off is that prolonged flickering can cause fatigue and, in rare cases, photosensitive epileptic reactions.

The present notebook focuses on the ERP/P300 approach; SSVEP is a natural extension for a subsequent notebook.

## 6. Data preparation

The MNE sample dataset is not a P300 speller recording, but it provides an ideal substrate for learning ERP-based classification: it contains stimulus-evoked responses from two modalities (auditory and visual) with clearly distinct temporal and spatial signatures. The classification task — discriminating **auditory** from **visual** single trials — employs the same feature extraction and classification techniques used in a P300 speller, where the task is to separate target from non-target responses.

In [ ]:
data_path = sample.data_path()
raw_fname = data_path / "MEG" / "sample" / "sample_audvis_filt-0-40_raw.fif"

raw = mne.io.read_raw_fif(raw_fname, preload=True)
events = mne.find_events(raw, stim_channel="STI 014")

# Two-class problem: auditory (left ear) vs visual (left field).
event_id = {"auditory": 1, "visual": 3}

epochs = mne.Epochs(
    raw, events, event_id,
    tmin=-0.1, tmax=0.8,              # 100 ms pre-stimulus, 800 ms post-stimulus
    picks="eeg",
    baseline=(None, 0),
    preload=True,
    reject=dict(eeg=150e-6),
)
epochs

In [ ]:
print("Trials per condition:")
for cond in event_id:
    print(f"  {cond}: {len(epochs[cond])}")

The two evoked responses, for reference (these were computed in notebook #1):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs["auditory"].average().plot(axes=axes[0], spatial_colors=True, show=False,
                                  titles=dict(eeg="Auditory ERP"))
epochs["visual"].average().plot(axes=axes[1], spatial_colors=True, show=False,
                                titles=dict(eeg="Visual ERP"))
plt.tight_layout()
plt.show()

Averaged across trials, the two modalities look clearly distinct — different peak latencies, different topographies. The central question of this notebook is: **can the same distinction be detected on a single-trial basis?**

## 7. Single-trial variability

Before classification, it is instructive to examine what individual epochs look like. The gap between the clean averaged ERP and the noisy single trial is the core challenge of ERP-based BCIs.

In [ ]:
# Plot 8 individual auditory trials at one electrode (EEG 021, central).
ch = "EEG 021"
ch_idx = epochs.ch_names.index(ch)

fig, axes = plt.subplots(2, 4, figsize=(14, 5), sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    trial = epochs["auditory"].get_data(copy=False)[i, ch_idx, :]
    ax.plot(epochs.times * 1e3, trial * 1e6, color="C0", alpha=0.8)
    ax.axvline(0, color="k", linestyle="--", linewidth=0.5)
    ax.set_title(f"Trial {i + 1}", fontsize=9)
    if i >= 4:
        ax.set_xlabel("Time (ms)")
    if i % 4 == 0:
        ax.set_ylabel("μV")
fig.suptitle(f"Individual auditory trials at {ch}", fontsize=11)
plt.tight_layout()
plt.show()

The N100 component (negative deflection near 100 ms) is sometimes visible in individual trials, but the signal-to-noise ratio is low. Some trials are dominated by noise or artifacts. This is the reality confronting a single-trial classifier — and the reason that P300 spellers typically average across multiple flash sequences before making a decision.

In [ ]:
# Overlay: individual trials (gray) vs average (colour).
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, cond, color in zip(axes, ["auditory", "visual"], ["C0", "C3"]):
    data = epochs[cond].get_data(copy=False)[:, ch_idx, :] * 1e6
    for trial in data:
        ax.plot(epochs.times * 1e3, trial, color="gray", alpha=0.08)
    avg = data.mean(axis=0)
    ax.plot(epochs.times * 1e3, avg, color=color, linewidth=2, label="Average")
    ax.axvline(0, color="k", linestyle="--", linewidth=0.5)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("μV")
    ax.set_title(f"{cond.capitalize()} — single trials vs average")
    ax.legend()
plt.tight_layout()
plt.show()

The average (coloured line) emerges from the cloud of individual traces. With ~70 trials per condition, the signal-to-noise gain from averaging is approximately √70 ≈ 8×. A classifier operating on single trials must cope with the full noise level.

## 8. ERP classification: the vectorised approach

The simplest ERP classification strategy proceeds as follows:

1. Take each epoch — a matrix of shape `(n_channels, n_times)`.
2. **Flatten** it into a single feature vector of length `n_channels × n_times`.
3. Standardise features to zero mean and unit variance.
4. Feed into a linear classifier (logistic regression or LDA).

This approach treats the full spatio-temporal waveform as the feature space. It is crude but effective — and it is the baseline against which more sophisticated methods are measured.

In [ ]:
X = epochs.get_data(copy=False)     # (n_epochs, n_channels, n_times)
y = epochs.events[:, -1]            # 1 = auditory, 3 = visual

# Recode labels to 0/1 for cleaner interpretation.
y = (y == 3).astype(int)            # 0 = auditory, 1 = visual

print("X shape:", X.shape)
print("y distribution:", np.bincount(y))

In [ ]:
# Vectoriser flattens (n_channels, n_times) → (n_channels * n_times,)
clf = make_pipeline(
    Vectorizer(),
    StandardScaler(),
    LogisticRegression(solver="liblinear", max_iter=1000),
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")

print(f"ROC AUC: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"Per-fold: {np.round(scores, 3)}")

An ROC AUC well above 0.5 (chance) is expected — typically 0.95+ for this particular task, because the auditory and visual ERPs are very distinct in both latency and topography.

In a real P300 speller the task is harder: target and non-target waveforms differ only in the presence or absence of a relatively small P300 component, and the class imbalance is extreme (≈ 1:5 or 1:10). Typical single-trial AUCs in P300 spelling range from 0.70 to 0.90.

### 8.1 Which features matter?

The classifier assigns a weight to each position in the flattened feature vector. Reshaping those weights back to `(n_channels, n_times)` reveals *where* and *when* the classifier looks.

In [ ]:
# Fit on all data for visualisation (not for evaluation — for that, CV is used above).
clf.fit(X, y)

# Extract the logistic regression coefficients.
coefs = clf.named_steps["logisticregression"].coef_[0]

# Undo the vectoriser reshape: recover (n_channels, n_times).
coef_map = coefs.reshape(X.shape[1], X.shape[2])

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(coef_map, aspect="auto", cmap="RdBu_r",
               extent=[epochs.times[0]*1e3, epochs.times[-1]*1e3, X.shape[1], 0],
               vmin=-np.percentile(np.abs(coef_map), 99),
               vmax=np.percentile(np.abs(coef_map), 99))
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Channel index")
ax.set_title("Classifier weights — channels × time")
plt.colorbar(im, ax=ax, label="Weight")
plt.show()

The resulting image shows which channel-time combinations carry discriminative information. High-weight regions should concentrate around the known ERP peak latencies (100–200 ms) and at channels over auditory or visual cortex. Low-weight regions (near zero) contribute little to the classification.

This visualisation is informative but somewhat difficult to interpret at 60 channels. The xDAWN spatial filter, introduced next, produces a much cleaner view.

## 9. xDAWN spatial filtering

**xDAWN** (Rivet et al., 2009) is the ERP analogue of CSP. Where CSP finds spatial filters maximising *variance difference* between classes, xDAWN finds spatial filters maximising the **signal-to-noise ratio of the evoked response**.

More precisely, xDAWN decomposes the multichannel signal into a small number of virtual channels (spatial components) that capture the largest possible fraction of the evoked response relative to the ongoing noise. The result is a low-dimensional, high-SNR representation of the ERP.

### 9.1 xDAWN vs CSP — when to use which

| | CSP (notebook #2) | xDAWN (this notebook) |
|---|---|---|
| **Signal type** | Induced power differences | Evoked waveform differences |
| **Optimises** | Variance ratio between classes | Evoked-response-to-noise ratio |
| **Feature extraction** | Log-variance of filtered signal | Temporal waveform of filtered signal |
| **Typical use** | Motor imagery | P300 spellers, ERP-based BCIs |

Both are supervised spatial filters, but they target different signal properties. Using CSP for ERPs or xDAWN for MI would be suboptimal.

In [ ]:
from mne.preprocessing import Xdawn

# Fit xDAWN with 3 components per condition.
xd = Xdawn(n_components=3)
xd.fit(epochs)

In [ ]:
# Display the spatial patterns identified by xDAWN.
fig = xd.apply(epochs)["auditory"].average().plot(
    spatial_colors=True, titles=dict(eeg="xDAWN-filtered auditory ERP")
)

The xDAWN-filtered signal concentrates the evoked response into a small number of components. The first component captures the dominant ERP pattern; subsequent components capture progressively weaker signals.

### 9.2 Classification with xDAWN-enhanced features

Applying xDAWN before vectorisation reduces the number of spatial dimensions from 60 to a handful, while preserving (and enhancing) the discriminative ERP signal. This typically improves classification, especially when trial counts are limited.

In [ ]:
# Apply xDAWN, then extract data for classification.
epochs_xd = xd.apply(epochs)
X_xd = epochs_xd.get_data(copy=False)

print("Original feature count:", X.shape[1] * X.shape[2])
print("xDAWN feature count:   ", X_xd.shape[1] * X_xd.shape[2])

clf_xd = make_pipeline(
    Vectorizer(),
    StandardScaler(),
    LogisticRegression(solver="liblinear", max_iter=1000),
)

scores_xd = cross_val_score(clf_xd, X_xd, y, cv=cv, scoring="roc_auc")
print(f"
ROC AUC (raw):   {scores.mean():.3f} ± {scores.std():.3f}")
print(f"ROC AUC (xDAWN): {scores_xd.mean():.3f} ± {scores_xd.std():.3f}")

For this dataset, where the auditory-visual distinction is already strong, the improvement from xDAWN may be marginal. In P300 speller data, where the signal difference is subtler and the dimensionality higher, xDAWN typically provides a substantial boost — reducing the number of averaged sequences needed for reliable spelling.

> **Note on data leakage.** In the above demonstration, xDAWN was fitted on the full dataset before cross-validation. For rigorous evaluation, xDAWN should be fitted only on the training fold. This simplification is acceptable here for clarity; production pipelines should embed xDAWN within the cross-validation loop.

## 10. Temporal decoding — when does the brain encode stimulus identity?

A distinctive advantage of ERP data is that the signal unfolds over time with millisecond resolution. Rather than asking *"Can the classifier tell the conditions apart?"*, a more informative question is: *"At which latencies does the brain encode information about the stimulus category?"*

The **temporal decoding** approach (also called "decoding over time") trains a separate classifier at each time point and measures its performance. The resulting accuracy-versus-time curve reveals the temporal dynamics of cortical processing.

In [ ]:
# A classifier is trained independently at each time sample.
clf_time = make_pipeline(StandardScaler(), LogisticRegression(solver="liblinear"))
sl = SlidingEstimator(clf_time, scoring="roc_auc", n_jobs=1)

scores_time = cross_val_multiscore(sl, X, y, cv=cv, n_jobs=1)
mean_scores_time = scores_time.mean(axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs.times * 1e3, mean_scores_time, color="C0")
ax.axhline(0.5, color="k", linestyle="--", linewidth=0.8, label="Chance")
ax.axvline(0, color="gray", linestyle=":", linewidth=0.8, label="Stimulus onset")
ax.fill_between(epochs.times * 1e3,
                scores_time.mean(0) - scores_time.std(0),
                scores_time.mean(0) + scores_time.std(0),
                alpha=0.15, color="C0")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("ROC AUC")
ax.set_title("Temporal decoding — auditory vs visual")
ax.legend()
plt.show()

Interpretation of the curve:

- **Before stimulus onset (t < 0)**: accuracy fluctuates around chance. No stimulus has occurred, so no information is present.
- **~50–100 ms**: accuracy begins to rise sharply, coinciding with the arrival of sensory input in primary cortex (the N100 latency).
- **~100–300 ms**: accuracy peaks and remains high, reflecting the temporal extent of the discriminative ERP components.
- **After ~400 ms**: accuracy may decline as the transient evoked response dissipates.

In a P300 speller context, the critical window is approximately 250–500 ms post-flash — the latency range of the P300 component.

### 10.1 Temporal generalisation matrix

An extension of temporal decoding asks: *"Does a classifier trained at time t also work at time t′?"* The **temporal generalisation matrix** (King & Dehaene, 2014) addresses this question. A square matrix is produced where entry (t, t′) gives the performance of a classifier trained at time t and tested at time t′.

- **Diagonal** = standard temporal decoding (train and test at the same time).
- **Off-diagonal** = generalisation across time points.

The pattern of the matrix reveals the nature of the underlying neural code:

- a **broad diagonal band** suggests a stable, sustained representation;
- a **narrow diagonal** suggests a rapidly changing sequence of representations;
- **off-diagonal patches** suggest that a representation reactivates at a later time.

In [ ]:
gen = GeneralizingEstimator(
    make_pipeline(StandardScaler(), LogisticRegression(solver="liblinear")),
    scoring="roc_auc", n_jobs=1
)

scores_gen = cross_val_multiscore(gen, X, y, cv=cv, n_jobs=1)
mean_gen = scores_gen.mean(axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    mean_gen, origin="lower", cmap="RdBu_r",
    extent=[epochs.times[0]*1e3, epochs.times[-1]*1e3,
            epochs.times[0]*1e3, epochs.times[-1]*1e3],
    vmin=0.3, vmax=0.7,
)
ax.set_xlabel("Test time (ms)")
ax.set_ylabel("Train time (ms)")
ax.set_title("Temporal generalisation matrix")
ax.axvline(0, color="k", linewidth=0.5)
ax.axhline(0, color="k", linewidth=0.5)
plt.colorbar(im, ax=ax, label="ROC AUC")
plt.show()

A broad square of high accuracy in the post-stimulus period indicates that the auditory/visual distinction is encoded by a relatively sustained cortical representation — a classifier trained at 150 ms still performs well when tested at 300 ms. This is expected: the sensory representation persists while the brain processes the stimulus.

In more transient paradigms (e.g., rapid serial visual presentation), the matrix narrows to a thin diagonal, indicating that the neural code changes rapidly over time.

## 11. From this analysis to a P300 speller

The techniques developed above translate directly to a functional P300 speller. The mapping is summarised below.

### 11.1 What changes

| This notebook | P300 speller |
|---|---|
| Task: auditory vs visual | Task: target flash vs non-target flash |
| Balanced classes (~50/50) | Imbalanced classes (~17% target, ~83% non-target) |
| One classification = one trial | One classification = one flash; letter determined by accumulating across flashes |
| Offline analysis | Online, real-time classification |

### 11.2 What stays the same

- **Feature space**: temporal ERP waveform, optionally enhanced by xDAWN.
- **Classifier**: regularised LDA or logistic regression — the same models used here.
- **Temporal window**: ~0 to 600 ms post-stimulus, consistent with the P300 latency.
- **Spatial channels**: full montage or a subset centred on Pz/Cz, where the P300 is maximal.

### 11.3 The accumulation step

A single flash produces a noisy single-trial classification. Multiple flash sequences are accumulated before selecting a letter:

1. For each row *i* and column *j*, maintain a cumulative score *S(i)* and *S(j)*.
2. After each flash, update the score of the flashed row/column with the classifier's output (probability or decision value).
3. After *K* sequences (typically 5–15), select the letter at the intersection of argmax(*S(row)*) and argmax(*S(col)*).

This accumulation is analogous to Bayesian evidence integration and trades speed for accuracy: more sequences → higher accuracy, slower spelling. Adaptive systems can stop early when the accumulated evidence exceeds a confidence threshold, balancing throughput and error rate dynamically.

### 11.4 Hardware requirements

A P300 speller requires:

- a visual display showing the spelling matrix,
- software that synchronises flash onsets with EEG trigger markers,
- an EEG amplifier with ≥ 8 channels (full 64-channel montages are common in research; consumer 8-channel systems are viable with xDAWN or Riemannian methods),
- a real-time classification pipeline running with latency < 100 ms.

Open-source implementations exist in **BCI2000** and **OpenViBE**; `mne-lsl` provides the streaming bridge for Python-based systems.

## 12. Summary — the ERP classification pipeline

```python
import mne
from mne.datasets import sample
from mne.preprocessing import Xdawn
from mne.decoding import Vectorizer, SlidingEstimator, cross_val_multiscore
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 1. Load and epoch
raw = mne.io.read_raw_fif(path, preload=True)
events = mne.find_events(raw, stim_channel="STI 014")
epochs = mne.Epochs(raw, events, event_id, tmin=-0.1, tmax=0.8,
                    picks="eeg", baseline=(None, 0), preload=True)

# 2. Extract arrays
X = epochs.get_data()
y = epochs.events[:, -1]

# 3a. Basic ERP classification
clf = make_pipeline(Vectorizer(), StandardScaler(),
                    LogisticRegression(solver="liblinear"))
scores = cross_val_score(clf, X, y, cv=5, scoring="roc_auc")

# 3b. xDAWN-enhanced classification
xd = Xdawn(n_components=3)
xd.fit(epochs)
X_xd = xd.apply(epochs).get_data()
scores_xd = cross_val_score(clf, X_xd, y, cv=5, scoring="roc_auc")

# 4. Temporal decoding
sl = SlidingEstimator(clf, scoring="roc_auc")
scores_time = cross_val_multiscore(sl, X, y, cv=5)
```

The three analyses — whole-window classification, xDAWN enhancement, and temporal decoding — form a comprehensive characterisation of any ERP-based BCI signal.

## 13. Further directions

1. **A real P300 speller dataset.** The MOABB library (`pip install moabb`) provides standardised access to multiple P300 speller recordings (BNCI2014_008, BNCI2014_009, EPFLP300). Applying the pipeline from this notebook to one of those datasets bridges the gap to a genuine BCI application.
2. **Riemannian classification.** The `pyriemann` library treats ERP covariance matrices as points on a Riemannian manifold; the resulting classifiers (MDM, tangent-space logistic regression) are state-of-the-art on most P300 benchmarks and require minimal hyperparameter tuning.
3. **SSVEP notebook.** The other stimulus-evoked paradigm exploits frequency-domain features (power at the flickering frequency) rather than temporal waveforms. It requires different feature extraction but the same cross-validation methodology.
4. **Adaptive classifiers.** In an online P300 speller, the classifier can update its parameters after each block using the user's feedback — the spelled letter confirms the correct target, providing labelled data for incremental retraining.
5. **Language models.** Combining P300 classifier output with character-level language model priors (n-gram or neural) substantially reduces the number of flash sequences required per letter, increasing communication speed.

The BCI textbook covers stimulus-evoked BCIs in section 9.1.4, with further discussion of communication applications in section 12.1.5. The SSVEP paradigm appears in section 9.1.4 as well and constitutes a natural candidate for notebook #4.